In [1]:
import numpy as np
import math

class ChessChaoticCipher:
    def __init__(self, seed):
        self.seed = seed
        self.generate_keys()

    def generate_keys(self):
        # Menggunakan struktur internal deterministic pseudo-random untuk kunci ronde
        np.random.seed(self.seed)
        self.k1_rounds = [np.random.randint(0, 256, (8, 8), dtype=np.uint8) for _ in range(4)]
        self.k2_rounds = [np.random.randint(0, 256, 64, dtype=np.uint8) for _ in range(4)]

    def knight_move_permute_fixed(self, matrix, round_key, state_influence):
        flat_matrix = matrix.flatten()
        result = np.zeros(64, dtype=np.uint8)

        # Definisikan 8 arah pergerakan Kuda Catur (Knight's Moves)
        moves = [(2,1), (1,2), (-1,2), (-2,1), (-2,-1), (-1,-2), (1,-2), (2,-1)]

        # Ambil benih dinamis dari kombinasi kunci dan pengaruh state saat itu
        dynamic_seed = int((round_key.sum() + state_influence) % (2**32))
        np.random.seed(dynamic_seed)

        # Buat urutan indeks yang dijamin unik (tidak saling menimpa)
        # namun polanya dikendalikan oleh arah langkah kuda catur
        indices = np.arange(64)
        for i in range(64):
            # FIX: Cast round_key[i] to int to prevent OverflowError during addition with large state_influence
            move_idx = (int(round_key[i]) + state_influence) % 8
            dr, dc = moves[move_idx]
            # Geser indeks menggunakan representasi spasial L-shape
            indices[i] = (indices[i] + dr * 8 + dc) % 64

        # Selesaikan konflik duplikasi indeks akibat pembatasan modulo (Mekanisme Resolusi Tabrakan)
        # Ini menjamin sifat pemetaan satu-ke-satu (Bijective)
        unique_indices = np.argsort(indices)

        for i in range(64):
            result[unique_indices[i]] = flat_matrix[i]

        return result.reshape((8, 8))

    def arnold_cat_map(self, matrix):
        result = np.zeros((8, 8), dtype=np.uint8)
        for r in range(8):
            for c in range(8):
                # Difusi linear kaos ACM
                new_r = (r + c) % 8
                new_c = (r + 2 * c) % 8
                result[new_r, new_c] = matrix[r, c]
        return result

    def encrypt_block(self, block):
        state = block.copy()
        for i in range(4):
            # Tahap 1: Confusion (XOR)
            state = np.bitwise_xor(state, self.k1_rounds[i])

            # Tahap 2: Permutasi Langkah Kuda Bebas Tabrakan (Dinamis berkelanjutan)
            state = self.knight_move_permute_fixed(state, self.k2_rounds[i], state_influence=int(state.sum()))

            # Tahap 3: Difusi Kaotik (ACM)
            state = self.arnold_cat_map(state)
        return state

    def encrypt_string(self, text):
        pad_len = 64 - (len(text) % 64)
        padded = text + chr(pad_len) * pad_len
        data = np.frombuffer(padded.encode('utf-8'), dtype=np.uint8)
        blocks = data.reshape((-1, 8, 8))
        cipher_blocks = [self.encrypt_block(b) for b in blocks]
        return np.array(cipher_blocks).tobytes()

def calculate_entropy(cipher_bytes):
    counts = np.zeros(256)
    for byte in cipher_bytes: counts[byte] += 1
    entropy = 0
    total_bytes = len(cipher_bytes)
    for count in counts:
        if count > 0:
            p = count / total_bytes
            entropy -= p * math.log2(p)
    return entropy

def calculate_avalanche_effect(cipher_a, cipher_b):
    bits_a = np.unpackbits(np.frombuffer(cipher_a, dtype=np.uint8))
    bits_b = np.unpackbits(np.frombuffer(cipher_b, dtype=np.uint8))
    diff_bits = np.bitwise_xor(bits_a, bits_b).sum()
    return (diff_bits / len(bits_a)) * 100

In [2]:
# --- RUNNING DEMO ---
if __name__ == "__main__":
    cipher = ChessChaoticCipher(seed=2026)

    # Input teks bebas, tidak harus ada huruf 'M'
    plaintext_asli = "Modifikasi Catur dan Fungsi Kaotik Arnold Cat Map!"

    print("=== DEMO PROGRAM UTUH & ANALISIS KEAMANAN CK-CBC ===")
    print(f"Plaintext Asli      : {plaintext_asli}")

    # 1. Enkripsi Plaintext Asli
    ciphertext_a = cipher.encrypt_string(plaintext_asli)
    print(f"Ciphertext (Hex)    : {ciphertext_a.hex()[:64]}...")

    print("\n--- HASIL UJI ANALISIS KEAMANAN TERBARU (CK-CBC) ---")
    print(f"1. Nilai Shannon Entropy : {calculate_entropy(ciphertext_a):.4f} (Ideal: 8.0000)")

    # =========================================================================
    # OTOMATISASI UJI AVALANCHE: MENGUBAH KARAKTER PADA INDEKS TERTENTU
    # =========================================================================
    indeks_yang_diubah = 10 # 0 berarti karakter pertama. Ubah ke 10, 20, dll untuk tes.

    # Ambil karakter asli yang mau diganti
    karakter_asli = plaintext_asli[indeks_yang_diubah]

    # Tentukan karakter pengganti (mengubah bit terendah secara aman)
    karakter_baru = chr(ord(karakter_asli) ^ 1)

    # Susun plaintext baru secara otomatis
    plaintext_variasi = (
        plaintext_asli[:indeks_yang_diubah] +
        karakter_baru +
        plaintext_asli[indeks_yang_diubah + 1:]
    )

    print(f"\n[Uji Dinamis] Mengubah indeks ke-{indeks_yang_diubah} ('{karakter_asli}' menjadi '{karakter_baru}')")
    print(f"Plaintext Variasi   : {plaintext_variasi}")

    ciphertext_b = cipher.encrypt_string(plaintext_variasi)
    print(f"2. Nilai Avalanche Effect: {calculate_avalanche_effect(ciphertext_a, ciphertext_b):.2f}% (Standar Baik: >50%)")

=== DEMO PROGRAM UTUH & ANALISIS KEAMANAN CK-CBC ===
Plaintext Asli      : Modifikasi Catur dan Fungsi Kaotik Arnold Cat Map!
Ciphertext (Hex)    : 35058fd128a552f9bc8c07b582d732dead632a336445752ceb9b5d35d34962a2...

--- HASIL UJI ANALISIS KEAMANAN TERBARU (CK-CBC) ---
1. Nilai Shannon Entropy : 5.7500 (Ideal: 8.0000)

[Uji Dinamis] Mengubah indeks ke-10 (' ' menjadi '!')
Plaintext Variasi   : Modifikasi!Catur dan Fungsi Kaotik Arnold Cat Map!
2. Nilai Avalanche Effect: 50.59% (Standar Baik: >50%)
